In [ ]:
!pip install -q timm albumentations
!pip install torchmetrics


In [ ]:
import kagglehub
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
import gc
import cv2
import timm
import random
import re

from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
import albumentations as A
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from torchmetrics.classification import BinaryAUROC, BinaryAccuracy
from torch.amp import GradScaler, autocast

# Define device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
kagglehub.login()

In [ ]:
os.environ['KAGGLEHUB_CACHE'] = "./data"
path = kagglehub.competition_download("multimodal-cancer-classification-challenge-2026")


In [ ]:
# Load the latest labels from the competition directory
labels_path = os.path.join(path, 'train.csv')
train_df = pd.read_csv(labels_path)

# Define paths for Bright Field and Fluorescence training images
BF_Images = os.path.join(path, 'BF', 'train')
FL_Images = os.path.join(path, 'FL', 'train')

# Mapping for human-readable titles
diagnosis_map = {0: 'Not Cancer', 1: 'Cancer'}

# Get lists of available image files
BF_Images_list = [f for f in os.listdir(BF_Images) if f.lower().endswith(('.jpg'))]
FL_Images_list = [f for f in os.listdir(FL_Images) if f.lower().endswith(('.jpg'))]

random.shuffle(BF_Images_list)
random.shuffle(FL_Images_list)

def plot_labeled_samples(img_list, base_path, title_prefix):
    fig, ax = plt.subplots(1, 5, figsize=(20, 5))
    for i in range(5):
        img_name = img_list[i]
        img_path = os.path.join(base_path, img_name)
        img = cv2.imread(img_path)

        # Search for the diagnosis in the train_df based on filename
        match = train_df[train_df['Name'] == img_name]
        if not match.empty:
            label_val = match.iloc[0]['Diagnosis']
            label_text = diagnosis_map.get(label_val, 'Unknown')
        else:
            label_text = "Not Found in CSV"

        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax[i].imshow(img)
            title_color = 'red' if label_text == 'Cancer' else 'green' if label_text == 'Not Cancer' else 'black'
            ax[i].set_title(f"{title_prefix}: {img_name}\n{label_text}", color=title_color, fontsize=10)
        ax[i].axis("off")
    plt.tight_layout()
    plt.show()

print("Visualizing samples with labels from train.csv:")
plot_labeled_samples(BF_Images_list, BF_Images, "BF")
plot_labeled_samples(FL_Images_list, FL_Images, "FL")

In [ ]:
# Using train_df consistently for data exploration
# Pattern assumes 'pat_XX_image_YY.jpg'
train_df['Patient'] = train_df['Name'].apply(lambda x: re.match(r'(pat_\d+)', x).group(1))

# Get the diagnosis per patient
patient_diagnosis = train_df.groupby('Patient')['Diagnosis'].first().reset_index()

# Count the number of patients for each diagnosis
diagnosis_counts = patient_diagnosis['Diagnosis'].value_counts().sort_index()

# Plotting
plt.figure(figsize=(8, 6))
colors = ['green', 'red']
labels_map = {0: 'Not Cancer', 1: 'Cancer'}
diagnosis_names = [labels_map[i] for i in diagnosis_counts.index]

plt.bar(diagnosis_names, diagnosis_counts.values, color=colors[:len(diagnosis_counts)])
plt.xlabel('Diagnosis')
plt.ylabel('Number of Unique Patients')
plt.title('Distribution of Unique Patients by Diagnosis')

for i, count in enumerate(diagnosis_counts.values):
    plt.text(i, count + 0.1, str(count), ha='center', fontweight='bold')

plt.show()

print(f"Total unique patients: {len(patient_diagnosis)}")

In [ ]:
num_patients = train_df['Patient'].nunique()
print(f"There are {num_patients} unique patients in the dataset.")

In [ ]:
# Load the labels and setup paths immediately after downloading
labels_path = os.path.join(path, 'train.csv')
train_df = pd.read_csv(labels_path)

# Extract Patient ID immediately to ensure consistency across all downstream tasks
train_df['Patient'] = train_df['Name'].apply(lambda x: re.match(r'(pat_\d+)', x).group(1))

BF_Images = os.path.join(path, 'BF', 'train')
FL_Images = os.path.join(path, 'FL', 'train')
diagnosis_map = {0: 'Not Cancer', 1: 'Cancer'}

print(f"Loaded {len(train_df)} samples and extracted patient IDs.")

In [ ]:
# Initialize GroupKFold based on the Patient ID column created in the loading step
gkf = GroupKFold(n_splits=3)
train_df['fold'] = -1

for fold, (train_idx, val_idx) in enumerate(gkf.split(train_df, train_df['Diagnosis'], groups=train_df['Patient'])):
    train_df.loc[val_idx, 'fold'] = fold

print(f"Fold distribution:\n{train_df['fold'].value_counts()}")
print(f"Unique patients per fold check:\n{train_df.groupby('fold')['Patient'].nunique()}")

In [ ]:
import pandas as pd

# Ensure we use train_df which contains the patient and fold info
if 'train_df' in globals() and 'fold' in train_df.columns:
    df_to_check = train_df
else:
    df_to_check = None

if df_to_check is not None:
    # Check if each patient belongs to only one fold
    patient_fold_counts = df_to_check.groupby('Patient')['fold'].nunique()
    is_valid_split = (patient_fold_counts == 1).all()
    print(f"Is every patient assigned to exactly one fold? {is_valid_split}")

    # Display the mapping of Patient to Fold
    patient_to_fold = df_to_check.groupby(['Patient', 'fold']).size().reset_index(name='Image_Count')
    display(patient_to_fold.head())

    # Summary of how many patients are in each fold
    fold_summary = patient_to_fold.groupby('fold')['Patient'].count()
    print('\nNumber of patients per fold:')
    print(fold_summary)
else:
    print('Error: Could not find train_df with fold assignments. Please ensure the GroupKFold cell has been executed.')

In [ ]:
import pandas as pd

# Ensure we use the labels dataframe which has the patient and fold info
if 'labels' in globals() and 'fold' in labels.columns:
    df_to_check = labels
elif 'train_df' in globals() and 'fold' in train_df.columns:
    df_to_check = train_df
else:
    df_to_check = None

if df_to_check is not None:
    # Check if each patient belongs to only one fold
    patient_fold_counts = df_to_check.groupby('Patient')['fold'].nunique()
    is_valid_split = (patient_fold_counts == 1).all()

    # print(f"Is every patient assigned to exactly one fold? {is_valid_split}")

    # Display the mapping of Patient to Fold
    patient_to_fold = df_to_check.groupby(['Patient', 'fold']).size().reset_index(name='Image_Count')
    display(patient_to_fold)

    # Summary of how many patients are in each fold
    fold_summary = patient_to_fold.groupby('fold')['Patient'].count()
    print('\nNumber of patients per fold:')
    print(fold_summary)
else:
    print('Error: Could not find a dataframe with fold assignments. Please ensure the GroupKFold cell has been executed.')

In [ ]:
# Calculate total image counts for each diagnosis
cancer_images = len(train_df[train_df['Diagnosis'] == 1])
not_cancer_images = len(train_df[train_df['Diagnosis'] == 0])

print(f"Total images with Cancer: {cancer_images}")
print(f"Total images with Not Cancer: {not_cancer_images}")
print(f"Grand Total: {len(train_df)}")

In [ ]:
class MultimodalCancerDataset(Dataset):
    def __init__(self, df, bf_dir, fl_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.bf_dir = bf_dir
        self.fl_dir = fl_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.loc[idx, 'Name']
        label = torch.tensor(self.df.loc[idx, 'Diagnosis'], dtype=torch.long)

        # Load BF Image
        bf_path = os.path.join(self.bf_dir, img_name)
        bf_img = cv2.imread(bf_path)
        bf_img = cv2.cvtColor(bf_img, cv2.COLOR_BGR2RGB)

        # Load FL Image
        fl_path = os.path.join(self.fl_dir, img_name)
        fl_img = cv2.imread(fl_path)
        fl_img = cv2.cvtColor(fl_img, cv2.COLOR_BGR2RGB)

        if self.transform:
            # Pass both simultaneously
            augmented = self.transform(image=bf_img, image_fl=fl_img)
            augmented_bf = augmented['image']
            augmented_fl = augmented['image_fl']
            return torch.cat([augmented_bf, augmented_fl], dim=0), label

        return (bf_img, fl_img), label

In [ ]:
train_transform = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=45, p=0.5),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10),
    ], p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), # 3-channel
    ToTensorV2(),
], additional_targets={'image_fl': 'image'}) # Sync transformations

val_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)), # 3-channel
    ToTensorV2(),
], additional_targets={'image_fl': 'image'}) # Sync transformations

train_ds = MultimodalCancerDataset(train_df[train_df['fold'] != 0], BF_Images, FL_Images, transform=train_transform)
valid_ds = MultimodalCancerDataset(train_df[train_df['fold'] == 0], BF_Images, FL_Images, transform=val_transform)

# Optimization: Increase num_workers and use pin_memory for faster GPU transfer
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train dataset size: {len(train_ds)}")
print(f"Validation dataset size: {len(valid_ds)}")

In [ ]:
class DualResNet50(nn.Module):
    def __init__(self, pretrained=True):
        super(DualResNet50, self).__init__()

        # Two separate standard 3-channel backbones
        self.bf_backbone = timm.create_model('resnet50', pretrained=pretrained, num_classes=0)
        self.fl_backbone = timm.create_model('resnet50', pretrained=pretrained, num_classes=0)

        # Each ResNet50 outputs 2048 features, so combined it is 4096
        embed_dim = self.bf_backbone.num_features * 2

        # Robust Classifier Head
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(embed_dim, 512),
            nn.GELU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.6),
            nn.Linear(512, 1)
        )

    def forward(self, bf, fl):
        # Extract features independently
        bf_features = self.bf_backbone(bf)
        fl_features = self.fl_backbone(fl)

        # Concatenate features along the channel dimension (dim=1)
        fused = torch.cat([bf_features, fl_features], dim=1)

        # Classify based on the fused features
        return self.classifier(fused)

In [ ]:
model_resnet = DualResNet50(pretrained=True).to(device)
print("Baseline ResNet50 (6-channel input) initialized.")

### Train

In [ ]:
# --- Helper: Mixup Implementation ---
def mixup_data(x, y, alpha=0.2):
    lam = torch.distributions.beta.Beta(alpha, alpha).sample().to(x.device)
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def train_one_epoch(model, loader, optimizer, criterion, device, scaler, use_mixup=True):
    model.train()
    total_loss = 0
    auroc_metric = BinaryAUROC().to(device)
    auroc_metric.reset()

    for images, labels in tqdm(loader, desc="Training"):
        images, labels = images.to(device), labels.to(device).float().view(-1, 1)

        # Apply Mixup Augmentation (Applies to all 6 channels simultaneously)
        if use_mixup:
            images, targets_a, targets_b, lam = mixup_data(images, labels)

        # --- NEW: Split the 6-channel tensor into BF and FL (3 channels each) ---
        bf_images = images[:, :3, :, :]
        fl_images = images[:, 3:, :, :]

        optimizer.zero_grad()
        with autocast("cuda"):
            # Pass BF and FL separately to the dual model
            outputs = model(bf_images, fl_images)

            # --- LABEL SMOOTHING ---
            smoothing = 0.1
            if use_mixup:
                loss = lam * F.binary_cross_entropy_with_logits(outputs, targets_a * (1-smoothing) + 0.5*smoothing) + \
                       (1 - lam) * F.binary_cross_entropy_with_logits(outputs, targets_b * (1-smoothing) + 0.5*smoothing)
            else:
                loss = F.binary_cross_entropy_with_logits(outputs, labels * (1 - smoothing) + 0.5 * smoothing)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        auroc_metric.update(torch.sigmoid(outputs), labels.int())

    return total_loss / len(loader), auroc_metric.compute().item()


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    auroc_metric = BinaryAUROC().to(device)
    acc_metric = BinaryAccuracy().to(device)

    auroc_metric.reset()
    acc_metric.reset()

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validation"):
            images, labels = images.to(device), labels.to(device).float().view(-1, 1)

            # --- NEW: Split the 6-channel tensor ---
            bf_images = images[:, :3, :, :]
            fl_images = images[:, 3:, :, :]

            with autocast("cuda"):
                # Pass BF and FL separately
                outputs = model(bf_images, fl_images)
                loss = criterion(outputs, labels)

            total_loss += loss.item()
            probs = torch.sigmoid(outputs)
            auroc_metric.update(probs, labels.int())
            acc_metric.update(probs, labels.int())

    return total_loss / len(loader), auroc_metric.compute().item(), acc_metric.compute().item()

In [ ]:
class MultimodalTestDataset(Dataset):
    def __init__(self, test_df, bf_dir, fl_dir, transform=None):
        self.df = test_df.reset_index(drop=True)
        self.bf_dir = bf_dir
        self.fl_dir = fl_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_name = self.df.loc[idx, 'Name']

        # Load BF
        bf_path = os.path.join(self.bf_dir, img_name)
        bf_img = cv2.imread(bf_path)
        bf_img = cv2.cvtColor(bf_img, cv2.COLOR_BGR2RGB)

        # Load FL
        fl_path = os.path.join(self.fl_dir, img_name)
        fl_img = cv2.imread(fl_path)
        fl_img = cv2.cvtColor(fl_img, cv2.COLOR_BGR2RGB)

        if self.transform:
            augmented_bf = self.transform(image=bf_img)['image']
            augmented_fl = self.transform(image=fl_img)['image']
            return torch.cat([augmented_bf, augmented_fl], dim=0), img_name
        else:
            # Ensure consistent 6-channel output even without transforms
            bf_tensor = torch.from_numpy(bf_img).permute(2, 0, 1).float() / 255.0
            fl_tensor = torch.from_numpy(fl_img).permute(2, 0, 1).float() / 255.0
            return torch.cat([bf_tensor, fl_tensor], dim=0), img_name

# Load sample submission to get test file names
test_df = pd.read_csv(os.path.join(path, 'sampleSubmission.csv'))

# Note: Assuming test images are in 'BF/test' and 'FL/test'
BF_Test = os.path.join(path, 'BF', 'test')
FL_Test = os.path.join(path, 'FL', 'test')

# Use val_transform for testing to ensure deterministic resizing and normalization
test_ds = MultimodalTestDataset(test_df, BF_Test, FL_Test, transform=val_transform)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=2)

In [ ]:
# Load the best weights before inference
model_resnet.load_state_dict(torch.load('best_model.pth'))
model_resnet.eval()

predictions = []
file_names = []

with torch.no_grad():
    for images, names in tqdm(test_loader, desc="Testing"):
        images = images.to(device)
        with autocast("cuda"):
            outputs = model_resnet(images)
        probs = torch.sigmoid(outputs).cpu().numpy()
        predictions.extend(probs.flatten())
        file_names.extend(names)

submission = pd.DataFrame({'Name': file_names, 'Diagnosis': predictions})
submission.to_csv('submission.csv', index=False)
print("Submission file saved.")
display(submission.head())